### LoRA rank ablation study
A short ablation study to get more insight of how the rank parameter affects the LoRA fine-tuning.

In [1]:
import pandas as pd
import sys
sys.path.append("..")
from src.previous_chapters import *

data_file_path = "sms_spam_collection/SMSSpamCollection.tsv"
df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
balanceded_df = create_balanced_dataset(df)
balanceded_df["Label"] = balanceded_df["Label"].map({"ham": 0, "spam": 1})

train_df, val_df, test_df = random_split(balanceded_df, 0.7, 0.1)
train_df.to_csv("train.csv", index=None)
val_df.to_csv("val.csv", index=None)
test_df.to_csv("test.csv", index=None)

In [2]:
import torch
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
train_dataset = SpamDataset("train.csv", max_length=None, tokenizer=tokenizer)
val_dataset   = SpamDataset("val.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)
test_dataset  = SpamDataset("test.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)

In [3]:
# Create DataLoaders for training, validation and testing
from torch.utils.data import DataLoader

num_workers = 0
batch_size  = 8

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False
)

In [4]:
# Verify batches, token size and total number of batches in each dataset
print("Train loader:")
for input_batch, target_batch in train_loader:
    pass
print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)
print("-"*80)
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

Train loader:
Input batch dimensions: torch.Size([8, 120])
Label batch dimensions torch.Size([8])
--------------------------------------------------------------------------------
130 training batches
19 validation batches
38 test batches


In [5]:
from ch05.gpt_download import *

CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG  = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();

File already exists and is up-to-date: gpt2\124M\checkpoint
File already exists and is up-to-date: gpt2\124M\encoder.json
File already exists and is up-to-date: gpt2\124M\hparams.json
File already exists and is up-to-date: gpt2\124M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2\124M\model.ckpt.index
File already exists and is up-to-date: gpt2\124M\model.ckpt.meta
File already exists and is up-to-date: gpt2\124M\vocab.bpe


In [6]:
# Make sure the model was loaded correctly
text_1 = "Every effort moves you"

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

Every effort moves you forward.

The first step is to understand the importance of your work


In [7]:
# Let's prepare the model for classification (similar to chapter 6)
torch.manual_seed(123)
num_classes = 2
model.out_head = torch.nn.Linear(in_features=768, out_features=num_classes)

In [8]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device: ", device)
model.to(device);

Device:  cuda


In [9]:
# Calculate inital classification accuracy of non-finetuned model
torch.manual_seed(123)
train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Training accuracy: 46.25%
Validation accuracy: 45.00%
Test accuracy: 48.75%


In [10]:
import math

class LoRALayer(torch.nn.Module):

    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = torch.nn.Parameter(torch.empty(in_dim, rank))
        torch.nn.init.kaiming_uniform_(self.A, a=math.sqrt(5)) # Initialize A matrix
        self.B = torch.nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha
        self.rank  = rank

    def forward(self, x):
        # Scaling the matrices
        x = (self.alpha / self.rank) * (x @ self.A @self.B)
        return x

class LinearWithLora(torch.nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora   = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)

def replace_linear_with_lora(model, rank, alpha):
    """Function that replaces standard linear layers with Lora linear layers in our GPT model"""
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):
            setattr(model, name, LinearWithLora(module, rank, alpha))
        else:
            # Recursively apply the function to child modules
            replace_linear_with_lora(module, rank, alpha)

In [11]:
# Freeze the original model parameters and replace them with lora layers
total_params_bef = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters before: {total_params_bef:,}")

for param in model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters before: {total_params:,}")

Total trainable parameters before: 124,441,346
Total trainable parameters before: 0


In [12]:
import copy

base_model = copy.deepcopy(model)

ranks = [1, 4, 8, 16, 32, 64]
for rank in ranks:
    torch.manual_seed(123)                # LoRA A matrices are randomly initialized
    run_model = copy.deepcopy(base_model)

    for p in run_model.parameters():      # freeze everything first
        p.requires_grad = False

    replace_linear_with_lora(run_model, rank=rank, alpha=16)

    total_params = sum(p.numel() for p in run_model.parameters() if p.requires_grad)
    print(f"Rank {rank}: {total_params:,} trainable parameters")

Rank 1: 166,658 trainable parameters
Rank 4: 666,632 trainable parameters
Rank 8: 1,333,264 trainable parameters
Rank 16: 2,666,528 trainable parameters
Rank 32: 5,333,056 trainable parameters
Rank 64: 10,666,112 trainable parameters


### Train the models
Training models using different ranks on 3 different seeds to get average accuracy scores over the training, validation and hold-out test sets. 

In [13]:
import copy
import csv
import os
import time

ranks = [1, 4, 8, 16, 32, 64]
seeds = [42, 123, 1337]
num_epochs = 5
csv_path = "lora_rank_ablation.csv"
fieldnames = ["rank", "seed", "total_params", "run_time_min",
              "train_accuracy", "val_accuracy", "test_accuracy"]

# Skip runs that are already saved
done = set()
if os.path.exists(csv_path):
    with open(csv_path, newline="") as f:
        done = {(int(r["rank"]), int(r["seed"])) for r in csv.DictReader(f)}

for rank in ranks:
    for seed in seeds:
        if (rank, seed) in done:
            continue

        start_time = time.time()
        torch.manual_seed(seed)

        run_model = copy.deepcopy(base_model)
        for p in run_model.parameters():          # freeze everything first
            p.requires_grad = False

        replace_linear_with_lora(run_model, rank=rank, alpha=16)
        run_model.to(device)                      

        total_params = sum(p.numel() for p in run_model.parameters() if p.requires_grad)

        optimizer = torch.optim.AdamW(
            [p for p in run_model.parameters() if p.requires_grad],
            lr=8e-4, weight_decay=0.1,
        )

        train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
            run_model, train_loader, val_loader, optimizer, device,
            num_epochs=num_epochs, eval_freq=50, eval_iter=5,
        )

        run_time = (time.time() - start_time) / 60

        row = {
            "rank": rank,
            "seed": seed,
            "total_params": total_params,
            "run_time_min": round(run_time, 2),
            "train_accuracy": calc_accuracy_loader(train_loader, run_model, device),
            "val_accuracy": calc_accuracy_loader(val_loader, run_model, device),
            "test_accuracy": calc_accuracy_loader(test_loader, run_model, device),
        }

        # Append one row per run so a crash never loses earlier results
        write_header = not os.path.exists(csv_path)
        with open(csv_path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            if write_header:
                writer.writeheader()
            writer.writerow(row)

        print(row)

        del run_model, optimizer
        torch.cuda.empty_cache()

Ep 1 (Step 000000): Train loss 2.395, Val loss 2.333
Ep 1 (Step 000050): Train loss 0.886, Val loss 0.944
Ep 1 (Step 000100): Train loss 0.986, Val loss 1.023
Training accuracy: 77.50% | Validation accuracy: 70.00%
Ep 2 (Step 000150): Train loss 0.600, Val loss 0.637
Ep 2 (Step 000200): Train loss 1.060, Val loss 1.149
Ep 2 (Step 000250): Train loss 0.781, Val loss 0.748
Training accuracy: 42.50% | Validation accuracy: 40.00%
Ep 3 (Step 000300): Train loss 0.661, Val loss 0.730
Ep 3 (Step 000350): Train loss 0.736, Val loss 0.704
Training accuracy: 62.50% | Validation accuracy: 57.50%
Ep 4 (Step 000400): Train loss 0.625, Val loss 0.712
Ep 4 (Step 000450): Train loss 0.644, Val loss 0.739
Ep 4 (Step 000500): Train loss 0.698, Val loss 0.744
Training accuracy: 65.00% | Validation accuracy: 45.00%
Ep 5 (Step 000550): Train loss 0.714, Val loss 0.697
Ep 5 (Step 000600): Train loss 0.591, Val loss 0.662
Training accuracy: 75.00% | Validation accuracy: 62.50%
{'rank': 1, 'seed': 42, 'total_

In [28]:
df = pd.read_csv("lora_rank_ablation.csv")

table = (
    df.groupby("rank")
      .agg(
          params_M=("total_params", lambda s: s.iloc[0] / 1e6),
          val_mean=("val_accuracy", "mean"),
          val_std=("val_accuracy", "std"),
          test_mean=("test_accuracy", "mean"),
          test_std=("test_accuracy", "std"),
          time_min=("run_time_min", "mean"),
      )
      .reset_index()
)

for col in ["val_mean", "val_std", "test_mean", "test_std"]:
    table[col] = table[col] * 100

table["Val acc (%)"] = table.apply(lambda r: f"{r.val_mean:.1f} ± {r.val_std:.1f}", axis=1)
table["Test acc (%)"] = table.apply(lambda r: f"{r.test_mean:.1f} ± {r.test_std:.1f}", axis=1)

final = table[["rank", "params_M", "Val acc (%)", "Test acc (%)", "time_min"]].round(2)
print(final)

   rank  params_M  Val acc (%) Test acc (%)  time_min
0     1      0.17  79.0 ± 18.9  80.9 ± 12.2      1.37
1     4      0.67   95.3 ± 3.4   94.9 ± 3.2      1.32
2     8      1.33   97.8 ± 1.5   96.0 ± 0.9      1.32
3    16      2.67   97.5 ± 0.4   96.9 ± 1.4      1.33
4    32      5.33   96.9 ± 1.0   97.2 ± 0.7      1.35
5    64     10.67   97.1 ± 0.4   96.1 ± 1.3      1.42


As can be seen in the table, rank = 1 is unstable with hig deviation, rank = 4 is better but the model still improves with higher rank. At rank = 8 it plateus meaning higher rank is not necessary for this task. 

Why LoRA:
LoRA does not make training shorter (on a small model such as this GPT-2) but instead saves on memory, the lower the rank the fewer gradients and AdamW states we need to save. 

Harder tasks usually require higher rank and what rank to use depends on how different the task is from what the model already knows. E.g. teaching the model a new language it has no understanding of would probably need a higher rank. More training data could also justify a higher rank